# MAPED, step by step

The stages behind `maped.run()`, one at a time. Every setting is shown with its default and a comment on what it does; change one and rerun the cell to see its effect. Edit `DATA_DIR`. Runs unchanged on an NVIDIA GPU (CUDA) or an Apple Silicon Mac (MPS).

In [ ]:
# Install once. The widget brings quantem.gpu with it; MAPED lives in quantem.
# pip install --extra-index-url https://test.pypi.org/simple "quantem-gpu>=0.0.1rc11" git+https://github.com/bobleesj/quantem.widget.git
# pip install git+https://github.com/bobleesj/quantem.git@main

In [ ]:
from quantem.gpu import io
from quantem.diffraction import MAPED

DATA_DIR = "/path/to/sample/tilts"

files = io.discover(DATA_DIR)
maped = MAPED(io.load(files), files)  # file names carry the tilt, used in plot titles
maped.show_tilts(
    plot_scale=None,                      # divide each bright-field image by this before plotting; None = as recorded
    show_detector_center=True,            # red cross at the geometric center of the detector, to see the beam offset
    bright_field_cmap="gray",             # matplotlib colormap: "gray", "viridis", "magma", "cividis"
    bright_field_contrast="linear_auto",  # presets clip to the 2nd-98th percentile; or {"lower_quantile": 0.01, "upper_quantile": 0.99}
    pattern_cmap="inferno",               # "inferno", "magma", "viridis", "gray"
    pattern_contrast="power_sqrt",        # "log_auto" for weak spots, "linear_auto" for the beam, or a dict as above with "power"
)


## 1. Beam center

Brightest pixel of each blurred mean diffraction pattern, in whole pixels. A marker for the plots; the alignment does not use it.

In [ ]:
maped.find_beam_center(
    blur=1,        # Gaussian blur in detector pixels before picking the brightest pixel (default 1); 0 = none
    centers=None,  # or (row, col) to set the center by hand, one tuple for all tilts or one per tilt
)

## 2. Detector alignment

Cross-correlates each tilt's mean diffraction pattern with the running average; one sub-pixel shift per tilt, `maped.diffraction_shifts`.

In [ ]:
maped.align_diffraction(
    border_taper=2,  # pixels faded to zero at the detector border so the border is not matched (default 2)
    precision=0.01,  # smallest shift step measured, in detector pixels; 1 = whole pixels only
)
maped.diffraction_shifts

## 3. Real-space alignment

Aligns each tilt's bright-field image to the average, rebuilds the average, repeats; one sub-pixel shift per tilt, `maped.real_space_shifts`. Matches edge strength rather than brightness, so brightness ramps between tilts do not pull it.

In [ ]:
maped.align_real_space(
    iterations=80,     # rounds of align-to-average; raise until doubling it changes nothing (default 20)
    hann_window=True,  # fade the scan border so the frame of the scan is not matched (default True)
    max_shift=5,       # largest scan shift expected, in pixels; pads against wrap-around, not a limit (default 5)
    edge_blur=2,       # width of an edge in scan pixels (default 2); None = match brightness instead of edges
    precision=0.01,    # smallest shift step measured, in scan pixels
)
maped.real_space_shifts

## 4. Merge

Weighted mean of the tilts at every scan position, each read at its shifted position. Computed in float32; stored as scaled 16-bit with the rounding error printed.

In [ ]:
merged = maped.merge(
    dtype="scaled_uint16",  # complete result as calibrated 16-bit codes on the GPU; "float32" only for a scan_region patch
    crop=True,              # keep only scan positions every tilt covers (default False, run() uses True); the box is printed
    scan_region=None,       # (row0, row1, col0, col1): exact float32 patch of at most 4096 positions instead
    save_to=None,           # path: stream regions to disk during the merge, then reopen; lower peak memory
    release_tilts=False,    # True: close the tilts once the merge no longer needs them
)

## View the merged result

Click or drag on the image to see the diffraction pattern at that position.


In [ ]:
from quantem.widget import Show4DSTEM

Show4DSTEM(merged)


## 5. Save and load

Commented out to avoid writing a second result on every run. Uncomment to use.

In [ ]:
# SAVE_PATH = "/path/to/results/merged_master.h5"
# io.save(SAVE_PATH, merged)       # scaled 16-bit codes and their calibration, as stored on the GPU
# maped.close()                    # release the merged result on the GPU before loading the file

In [ ]:
# saved = io.load(SAVE_PATH)       # reopens compressed on the GPU; saved[row, col] is a Torch tensor
# Show4DSTEM(saved)

## Same thing in one call

`run()` uses these defaults: `blur=1`, `border_taper=2`, `iterations=20`, `hann_window=True`, `max_shift=5`, `edge_blur=2`, `precision=0.01`, `dtype="scaled_uint16"`, `crop=True`. Any of them is a keyword of `run()`.

In [ ]:
merged = maped.run(iterations=80, show=False)